<img class="nvidia-header-light" src="images/nvidia_header_black.png" style="margin-left: -30px; width: 300px; float: left;">
<img class="nvidia-header-dark" src="images/nvidia_header_white.png" style="margin-left: -30px; width: 300px; float: left;">

# Part 3 — World Generation with Cosmos 3

In Part 2 we used Cosmos 3 as a vision-language model that *understands and critiques* scenes. This notebook uses Cosmos 3 as a **world foundation model (WFM)** that *generates* them — producing photorealistic, physically plausible video for training Physical AI models. Together, the reasoner and the world model close the loop of a synthetic-data augmentation pipeline.

Let's explore Cosmos 3 generation capabilities, running on the native **Cosmos Framework** (`python -m cosmos_framework.scripts.inference`):

- **Generation** — from a text prompt (text-to-video) or a starting image (image-to-video), Cosmos 3 synthesizes realistic video consistent with physical dynamics like motion, lighting, and object interactions.
- **Augmentation** — Cosmos 3 transforms an existing video's visual attributes (materials, colors, surfaces, lighting) while preserving the underlying scene structure, actions, and physics. This is how one real recording becomes many on-distribution variations.

This notebook runs in the **"Cosmos 3 (framework)"** kernel. If the kernel selector (top right) shows something else, switch it to *Cosmos 3 (framework)*.

## How this notebook fits the pipeline

The augmentation prompts you generated with the reasoner in Part 2 are exactly what drives generation here. After generating, you can route the outputs back through the Part 2 reasoner (e.g. its physical common-sense checks) to validate them, and the resulting diverse, physically grounded clips become training data for the perception and **action** models in Part 4.

## 1. Environment

The Cosmos Framework is **pre-installed** in this lab at `/opt/cosmos3-framework` (its virtual environment is the kernel you are using), so there is no clone or `uv sync` step. The cell below points the notebook at that checkout and the cookbook's sample assets, and selects the GPU.

By default we use a **single H100** (`CUDA_VISIBLE_DEVICES=1`) so generation can run while the Part 2 Reasoner NIM keeps GPU 0. To generate faster on both GPUs, stop the NIM from a host terminal (`docker compose -f task1/docker-compose.yml stop cosmos3-reasoner`) and set `CUDA_VISIBLE_DEVICES=0,1` and `COSMOS3_NUM_GPUS=2` below.

In [ ]:
import os
import json
import socket
from pathlib import Path

def free_local_port():
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.bind(("127.0.0.1", 0))
        return str(s.getsockname()[1])

# Cosmos cookbook checkout (assets + specs) and the pre-installed framework.
COSMOS_ROOT = Path(os.environ.get("COSMOS_ROOT", "/dli/task/cosmos")).resolve()
COSMOS3_REPO = Path(os.environ.get("COSMOS3_REPO", "/opt/cosmos3-framework")).resolve()
COSMOS3_VENV = COSMOS3_REPO / ".venv"

AUDIOVISUAL_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "audiovisual"
TRANSFER_ROOT = COSMOS_ROOT / "cookbooks" / "cosmos3" / "generator" / "transfer"
# Everything this notebook generates (payloads + videos) goes under <repo>/outputs/notebook3.
DLI_OUTPUTS = Path(os.environ.get("DLI_OUTPUTS", Path("/dli/task").resolve().parent / "outputs"))
OUTPUT_ROOT = Path(os.environ.get("COSMOS3_OUTPUT_ROOT", DLI_OUTPUTS / "notebook3"))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# GPU selection (see the note above to use both GPUs).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")
os.environ.setdefault("COSMOS3_NUM_GPUS", "1")
os.environ.setdefault("COSMOS3_MASTER_ADDR", "127.0.0.1")
os.environ.setdefault("COSMOS3_MASTER_PORT", free_local_port())
os.environ.setdefault("HF_HOME", str(Path.home() / ".cache" / "huggingface"))

# Export for the %%bash inference cells.
os.environ["COSMOS3_REPO"] = str(COSMOS3_REPO)
os.environ["COSMOS3_CHECKPOINT_PATH"] = os.environ.get("COSMOS3_CHECKPOINT_PATH", "Cosmos3-Nano")

for p in [COSMOS3_REPO, AUDIOVISUAL_ROOT, TRANSFER_ROOT]:
    assert p.exists(), f"missing: {p}"

print("Cosmos Framework:", COSMOS3_REPO)
print("Cookbook assets:  ", AUDIOVISUAL_ROOT)
print("Output root:      ", OUTPUT_ROOT)
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"], "| GPUs:", os.environ["COSMOS3_NUM_GPUS"])
print("checkpoint:", os.environ["COSMOS3_CHECKPOINT_PATH"])

In [ ]:
# Confirm the framework venv sees the GPU(s). (This kernel IS the framework venv.)
import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available(), "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  device {i}: {torch.cuda.get_device_name(i)}")

### Helpers

Cosmos 3 generation takes a **structured JSON prompt** (a rich, attribute-by-attribute scene description) rather than a single sentence — the cookbook ships several ready-made prompts under `assets/prompts/`. The helpers below assemble an inference *payload* from a prompt asset and display the resulting media.

In [ ]:
import base64, html, json
from IPython.display import HTML, Image, display, JSON

# Fixed sampling settings shared by the generation examples.
FIXED_SAMPLING = {
    "num_steps": 35, "guidance": 6.0, "shift": 10.0, "fps": 24,
    "num_frames": 189, "resolution": "720", "aspect_ratio": "16,9", "seed": 0,
}

def _compact(path):
    return json.dumps(json.loads(Path(path).read_text()), ensure_ascii=True, separators=(",", ":"))

def create_payload(name, mode, prompt_rel, image_rel=None, enable_sound=False):
    """Write a Cosmos Framework inference payload and return (payload_path, output_dir)."""
    payload_dir = OUTPUT_ROOT / "payloads"
    output_dir = OUTPUT_ROOT / name
    payload_dir.mkdir(parents=True, exist_ok=True)
    output_dir.mkdir(parents=True, exist_ok=True)

    prompt_path = AUDIOVISUAL_ROOT / prompt_rel
    negative = ""
    if mode != "text2image":
        negative = _compact(AUDIOVISUAL_ROOT / f"assets/negative_prompts/{mode}/neg_prompt.json")

    payload = {"model_mode": mode, "name": name, "prompt": _compact(prompt_path),
               "negative_prompt": negative, "enable_sound": enable_sound, **FIXED_SAMPLING}
    if mode == "text2image":
        payload["num_frames"] = 1

    payload_path = payload_dir / f"{name}.json"
    if mode == "image2video":
        image_path = (AUDIOVISUAL_ROOT / image_rel).resolve()
        payload["vision_path"] = os.path.relpath(image_path, payload_path.parent)
        display(Image(filename=str(image_path), width=420))

    payload_path.write_text(json.dumps(payload, indent=2) + "\n")
    os.environ[f"{name.upper()}_INPUT"] = str(payload_path)
    os.environ[f"{name.upper()}_OUTPUT"] = str(output_dir)
    print("payload:", payload_path)
    print("output: ", output_dir)
    print(json.dumps({k: payload[k] for k in ("model_mode", "num_steps", "guidance",
          "fps", "num_frames", "resolution", "aspect_ratio", "seed")}, indent=2))
    return payload_path, output_dir

def display_video(path, width=640):
    data = base64.b64encode(Path(path).read_bytes()).decode("ascii")
    display(HTML(f'<video controls playsinline width="{width}" style="background:#000">'
                 f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'
                 f'<div style="font-family:monospace;font-size:12px">{html.escape(str(path))}</div>'))

def view_run(output_dir):
    output_dir = Path(output_dir)
    media = [p for p in sorted(output_dir.rglob("*.mp4")) if not p.name.endswith("_preview.mp4")]
    imgs = sorted(output_dir.rglob("*.jpg")) + sorted(output_dir.rglob("*.png"))
    if not media and not imgs:
        print("No generated media under", output_dir); return
    for p in media:
        print(f"{p} ({p.stat().st_size // 1024} KB)"); display_video(p)
    for p in imgs:
        print(f"{p} ({p.stat().st_size // 1024} KB)"); display(Image(filename=str(p), width=640))

def view_t2v_prompt(prompt_file):
    prompt_file = AUDIOVISUAL_ROOT / prompt_file
    with open(prompt_file, "r") as f:
        data = json.load(f)
    display(JSON(data, expanded=True))

print("helpers ready")

## 2. Text-to-Video generation

We start from a structured text prompt (a robot working in a kitchen) and generate a short clip. The defaults produce a 189-frame (~8 s) 720p video in 35 denoising steps.

> Generation is compute-intensive: expect a few minutes on a single H100. To iterate faster you can lower `num_frames`, lower `num_steps`, or drop `resolution` to `"256"` in `FIXED_SAMPLING`. The first run also downloads the `Cosmos3-Nano` weights from Hugging Face (gated — make sure you completed notebook 01).

In [ ]:
t2v_prompt_file = "assets/prompts/text2video/robot_kitchen.json"
view_t2v_prompt(t2v_prompt_file)

t2v_payload, t2v_output = create_payload(
    "t2v_robot_kitchen", "text2video",
    "assets/prompts/text2video/robot_kitchen.json")

os.environ["T2V_INPUT"] = str(t2v_payload)
os.environ["T2V_OUTPUT"] = str(t2v_output)

In [ ]:
%%bash
set -euo pipefail
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
.venv/bin/torchrun \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" --master-port="$COSMOS3_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$T2V_INPUT" \
  -o "$T2V_OUTPUT" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed=0

In [ ]:
view_run(t2v_output)

## 3. Image-to-Video generation

Image-to-video anchors the first frame on a real image and lets Cosmos 3 animate it forward in a physically plausible way. Here we start from a still of a car on a road and generate motion.

In [ ]:
i2v_image_input = "assets/images/image2video/car_driving.jpg"

i2v_payload, i2v_output = create_payload(
    "i2v_car_driving", "image2video",
    "assets/prompts/image2video/car_driving.json",
    i2v_image_input)

os.environ["I2V_INPUT"] = str(i2v_payload)
os.environ["I2V_OUTPUT"] = str(i2v_output)

In [ ]:
%%bash
set -euo pipefail
cd "$COSMOS3_REPO"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
.venv/bin/torchrun \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" --master-port="$COSMOS3_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i "$I2V_INPUT" \
  -o "$I2V_OUTPUT" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed=0

In [ ]:
view_run(i2v_output)

## 4. Data augmentation

In the examples so far, we used Cosmos 3 to generate videos from scratch, based on a prompt and initial image. Cosmos 3 can also be used to augment existing video datasets by taking a video and re-rendering its appearance while preserving structure, motion, and physics. This helps turn one recording into many on-distribution variations. It is conditioned on a **control video** (one of edge, blur, depth, segmentation, or world-scenario maps) plus a prompt:

| Control | Preserves | Good for |
| --- | --- | --- |
| **Edge** | shapes, layout, fine detail | material / texture changes |
| **Depth** | 3D structure, spatial scale | relighting, surface changes |
| **Segmentation** | object/region identity | replacing objects or backgrounds |
| **Blur (vis)** | colors, lighting, overall look | subtle appearance edits |

The repo ships pre-computed control videos and matching prompts under `transfer/assets/`, and Cosmos Framework input specs under `transfer/specs/`. We will run the **edge** example. The prompt guides *what changes*; the edge map keeps the structure fixed.

In [ ]:
# Inspect the edge transfer spec, its prompt, and preview the control video.
edge_spec = TRANSFER_ROOT / "specs" / "edge.json"
spec = json.loads(edge_spec.read_text())
print(json.dumps(spec, indent=2))

prompt_json = json.loads((TRANSFER_ROOT / "assets" / "edge" / "prompt.json").read_text())
print("\nPrompt (first subject):")
print(json.dumps(prompt_json.get("subjects", [prompt_json])[0], indent=2)[:600])

os.environ["TRANSFER_DIR"] = str(TRANSFER_ROOT)
os.environ["TRANSFER_OUTPUT"] = str(OUTPUT_ROOT / "transfer_edge")
display_video(TRANSFER_ROOT / "assets" / "edge" / "control_edge.mp4", width=480)

Now run the transfer. Transfer inference is selected automatically because the spec contains a control (`edge`) block. The output video's frame count and geometry come from the control video.

> Transfer over a full clip takes several minutes on one H100. Edge is the lightest control to start with; once it works, try `specs/depth.json` or `specs/seg.json` by changing `-i` below.

In [ ]:
%%bash
set -euo pipefail
cd "$TRANSFER_DIR"
CUDA_VISIBLE_DEVICES="$CUDA_VISIBLE_DEVICES" LD_LIBRARY_PATH= \
"$COSMOS3_REPO/.venv/bin/torchrun" \
  --nproc-per-node="$COSMOS3_NUM_GPUS" \
  --master-addr="$COSMOS3_MASTER_ADDR" --master-port="$COSMOS3_MASTER_PORT" \
  -m cosmos_framework.scripts.inference \
  --parallelism-preset=throughput \
  -i specs/edge.json \
  -o "$TRANSFER_OUTPUT" \
  --checkpoint-path "$COSMOS3_CHECKPOINT_PATH" \
  --seed 2026

In [ ]:
view_run(OUTPUT_ROOT / "transfer_edge")

## Closing the loop

You generated new videos from a text prompt and from an image, and augmented an existing clip with a control modality — all with one Cosmos 3 model. In a full pipeline these outputs would be screened by the Part 2 reasoner's physical common-sense checks and then used, alongside the **action** data in Part 4, to train robust Physical AI perception and control models.

Try experimenting with **dramatic lighting shifts** or new materials in the prompts to create diverse data, and explore the other control modalities for video-to-video transfer.

## Resources

- [Cosmos 3 cookbook — Generator (audiovisual)](https://github.com/NVIDIA/cosmos/tree/main/cookbooks/cosmos3/generator/audiovisual)
- [Cosmos 3 cookbook — Generator (transfer)](https://github.com/NVIDIA/cosmos/tree/main/cookbooks/cosmos3/generator/transfer)
- [Cosmos Framework](https://github.com/NVIDIA/cosmos-framework)

<br clear="all">
<hr>
<img class="nvidia-header-light" src="images/nvidia_header_black.png" style="margin-left: -30px; width: 300px; float: left;">
<img class="nvidia-header-dark" src="images/nvidia_header_white.png" style="margin-left: -30px; width: 300px; float: left;">